# HAVEN AI — Stable Diffusion + ControlNet server (Colab + ngrok)

Run this notebook on a **GPU** Colab runtime. It exposes a FastAPI endpoint at `POST /generate` and prints a public ngrok URL. Paste that URL into `backend/.env` as `SD_CONTROLNET_URL=...` and restart the backend.

Contract:
```json
POST /generate
{
  "prompt": "...",
  "negative_prompt": "...",
  "control_image_b64": "<png>",
  "num_inference_steps": 25,
  "guidance_scale": 7.5,
  "controlnet_conditioning_scale": 1.1,
  "seed": 1234
}
-> { "image_b64": "<png>" }
```

In [ ]:
!pip -q install diffusers==0.30.3 transformers==4.44.2 accelerate==0.34.2 safetensors==0.4.5 \
               fastapi==0.115.0 uvicorn==0.30.6 pyngrok==7.2.0 nest-asyncio==1.6.0 pillow

In [ ]:
# --- Load Stable Diffusion + ControlNet ---
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

# MLSD ControlNet is tuned for straight lines — perfect for architectural drawings.
controlnet = ControlNetModel.from_pretrained(
    'lllyasviel/sd-controlnet-mlsd',
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    'runwayml/stable-diffusion-v1-5',
    controlnet=controlnet,
    safety_checker=None,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
).to(device)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe.enable_attention_slicing()
print('pipeline loaded')

In [ ]:
# --- FastAPI server ---
import base64, io
from PIL import Image
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class Req(BaseModel):
    prompt: str
    negative_prompt: str = ''
    control_image_b64: str
    num_inference_steps: int = 25
    guidance_scale: float = 7.5
    controlnet_conditioning_scale: float = 1.1
    seed: int = 1234

@app.get('/health')
def health():
    return {'status': 'ok', 'device': device}

@app.post('/generate')
def generate(req: Req):
    img_bytes = base64.b64decode(req.control_image_b64)
    ctrl = Image.open(io.BytesIO(img_bytes)).convert('RGB').resize((768, 768))
    generator = torch.Generator(device=device).manual_seed(req.seed)
    out = pipe(
        prompt=req.prompt,
        negative_prompt=req.negative_prompt,
        image=ctrl,
        num_inference_steps=req.num_inference_steps,
        guidance_scale=req.guidance_scale,
        controlnet_conditioning_scale=req.controlnet_conditioning_scale,
        generator=generator,
    ).images[0]
    buf = io.BytesIO()
    out.save(buf, format='PNG')
    return {'image_b64': base64.b64encode(buf.getvalue()).decode('ascii')}

In [ ]:
# --- Expose via ngrok ---
# 1. Get a free authtoken at https://dashboard.ngrok.com/get-started/your-authtoken
# 2. Paste it here.
NGROK_AUTHTOKEN = ''  # <-- paste your ngrok authtoken

from pyngrok import ngrok, conf
import nest_asyncio, uvicorn, threading

conf.get_default().auth_token = NGROK_AUTHTOKEN
public_url = ngrok.connect(8000, 'http').public_url
print('SD_CONTROLNET_URL =', public_url)
print('Paste that into backend/.env and restart the backend.')

nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000), daemon=True).start()